# Task 03 — Data Ingestion, Cleaning & Preprocessing with Pandas
Synthetic retail practice data. Run the cells top to bottom.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_path = Path('retail-orders-raw-12000.csv')
out_path = Path('clean_dataset.csv')
df = pd.read_csv(raw_path)
print('Raw shape:', df.shape)
display(df.head())


## 1. Data quality profile
Inspect data types, missing values, duplicates and distributions before cleaning.

In [ ]:
profile = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing_count': df.isna().sum(), 'missing_pct': (df.isna().mean()*100).round(2), 'unique_values': df.nunique(dropna=True)})
display(profile)
print('Duplicate rows:', int(df.duplicated().sum()))
display(df.describe(include='all').T)


## 2. Clean and standardize
Missing discount values are assumed to be 0 for this exercise. Invalid dates, missing required fields, invalid quantity/price and discounts outside 0–100 are excluded.

In [ ]:
clean = df.copy()
raw_rows = len(clean)
clean['order_date'] = pd.to_datetime(clean['order_date'], errors='coerce')
for col in ['quantity','unit_price','discount_pct']:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')
for col in ['customer_segment','city','category','payment_status']:
    clean[col] = clean[col].astype('string').str.strip()
for col in ['customer_segment','city','category','payment_status']:
    clean[col] = clean[col].str.title()
clean['discount_pct'] = clean['discount_pct'].fillna(0)
clean = clean.drop_duplicates()
clean = clean.dropna(subset=['order_id','order_date','customer_segment','city','category','quantity','unit_price','payment_status'])
clean = clean[(clean['quantity'] > 0) & (clean['quantity'] % 1 == 0)]
clean = clean[(clean['unit_price'] > 0) & clean['discount_pct'].between(0,100)]
clean['quantity'] = clean['quantity'].astype(int)


## 3. Feature engineering
Gross sales = quantity × unit price. Net sales = gross sales − discount amount.

In [ ]:
clean['gross_sales'] = (clean['quantity'] * clean['unit_price']).round(2)
clean['discount_amount'] = (clean['gross_sales'] * clean['discount_pct'] / 100).round(2)
clean['net_sales'] = (clean['gross_sales'] - clean['discount_amount']).round(2)
clean['order_month'] = clean['order_date'].dt.to_period('M').astype(str)
clean['order_date'] = clean['order_date'].dt.strftime('%Y-%m-%d')


## 4. Validate and export
Assertions check core rules before saving the output CSV.

In [ ]:
assert clean['order_id'].notna().all()
assert clean['quantity'].gt(0).all()
assert clean['unit_price'].gt(0).all()
assert clean['discount_pct'].between(0,100).all()
clean.to_csv(out_path, index=False)
print(f'Raw rows: {raw_rows:,}')
print(f'Clean rows: {len(clean):,}')
print(f'Removed rows: {raw_rows-len(clean):,}')
print('Saved:', out_path.resolve())
display(clean.head())
